<a href="https://colab.research.google.com/github/Aastha210/manual-transformer-encoder/blob/main/manual_transformer_encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manual Transfer Encoder
(Without ML Libraries)

### Step 1: Raw Text

In [1]:
T0 = "OMG!!! AI students rrr brillianttt 😊"
print(T0)

OMG!!! AI students rrr brillianttt 😊


### Step 2: preprocessing

In [7]:
def preprocess(text):
    text = text.lower()
    text = text.replace("omg", "")
    text = text.replace("rrr", "are")
    text = text.replace("brillianttt", "brilliant")
    cleaned = ""
    for ch in text:
        if ("a" <= ch <= "z") or ch == " ":
            cleaned += ch
    words = cleaned.split()
    return " ".join(words)
T1 = preprocess(T0)
print("After Preprocessing:")
print(T1)

After Preprocessing:
ai students are brilliant


### Step 3: Tokenization

In [8]:
tokens = T1.split()
print("Tokens:")
for i in range(len(tokens)):
    print("t" + str(i + 1), "=", tokens[i])

Tokens:
t1 = ai
t2 = students
t3 = are
t4 = brilliant


### Step 4: Token IDs

In [12]:
vocab = {
    "ai": 101,
    "students": 205,
    "are": 306,
    "brilliant": 412
}
token_ids = []
for token in tokens:
    token_ids.append(vocab[token])
print("Token IDs:")
for i in range(len(tokens)):
    print(tokens[i], ":", token_ids[i])

Token IDs:
ai : 101
students : 205
are : 306
brilliant : 412


### Step 5: Embedding Lookup

In [16]:
embedding_table = {
    101: [0.20, 0.40, 0.10, 0.50],
    205: [0.60, 0.10, 0.80, 0.30],
    306: [0.10, 0.70, 0.20, 0.90],
    412: [0.90, 0.30, 0.60, 0.20]
}
X = []
for token_id in token_ids:
    X.append(embedding_table[token_id])
print("Embedding Matrix X:")
for i in range(len(X)):
    print(X[i], ":", tokens[i])

Embedding Matrix X:
[0.2, 0.4, 0.1, 0.5] : ai
[0.6, 0.1, 0.8, 0.3] : students
[0.1, 0.7, 0.2, 0.9] : are
[0.9, 0.3, 0.6, 0.2] : brilliant


Basic Mathematical Functions

In [18]:
def sine(x):
    # Reduce x approximately to -pi to pi
    pi = 3.141592653589793
    while x > pi:
        x -= 2 * pi
    while x < -pi:
        x += 2 * pi
    # Taylor series for sin(x)
    result = 0.0
    term = x
    for n in range(10):
        result += term
        numerator = -x * x
        denominator = (2 * n + 2) * (2 * n + 3)
        term = term * numerator / denominator
    return result

def cosine(x):
    # Reduce x approximately to -pi to pi
    pi = 3.141592653589793
    while x > pi:
        x -= 2 * pi
    while x < -pi:
        x += 2 * pi
    # Taylor series for cos(x)
    result = 0.0
    term = 1.0
    for n in range(10):
        result += term
        numerator = -x * x
        denominator = (2 * n + 1) * (2 * n + 2)
        term = term * numerator / denominator
    return result

def exponential(x):
    # e^x using Taylor expansion
    # For large negative values, calculate using reciprocal
    if x < 0:
        return 1.0 / exponential(-x)
    result = 1.0
    term = 1.0
    for n in range(1, 40):
        term = term * x / n
        result += term
    return result

def square_root(x):
    # Newton-Raphson method
    if x == 0:
        return 0.0
    guess = x
    for _ in range(20):
        guess = (guess + x / guess) / 2
    return guess

def matrix_multiply(A, B):
    rows_A = len(A)
    cols_A = len(A[0])
    rows_B = len(B)
    cols_B = len(B[0])
    if cols_A != rows_B:
        raise ValueError("Matrix dimensions do not match")
    result = []
    for i in range(rows_A):
        row = []
        for j in range(cols_B):
            total = 0.0
            for k in range(cols_A):
                total += A[i][k] * B[k][j]
            row.append(total)
        result.append(row)
    return result

def print_matrix(matrix, digits=4):
    for row in matrix:
        print(
            "[",
            " ".join(f"{value:.{digits}f}" for value in row),
            "]"
        )

### Step 6: Positional Encoding

In [19]:
d_model = 4
PE = []
for pos in range(len(tokens)):
    row = []
    for i in range(d_model):
        if i % 2 == 0:
            # PE(pos, i) = sin(pos / 10000^(2i/d))
            angle = pos / (10000 ** (i / d_model))
            value = sine(angle)
        else:
            # PE(pos, i) = cos(pos / 10000^(2i/d))
            angle = pos / (10000 ** ((i - 1) / d_model))
            value = cosine(angle)
        row.append(value)
    PE.append(row)
print("Positional Encoding:")
print_matrix(PE)

Positional Encoding:
[ 0.0000 1.0000 0.0000 1.0000 ]
[ 0.8415 0.5403 0.0100 1.0000 ]
[ 0.9093 -0.4161 0.0200 0.9998 ]
[ 0.1411 -0.9900 0.0300 0.9996 ]


In [21]:
Z = []
for i in range(len(X)):
    row = []
    for j in range(d_model):
        row.append(X[i][j] + PE[i][j])
    Z.append(row)
print("Final Transformer Input Z = X + PE:")
for i in range(len(Z)):
    print([round(x, 4) for x in Z[i]], ":", tokens[i])

Final Transformer Input Z = X + PE:
[0.2, 1.4, 0.1, 1.5] : ai
[1.4415, 0.6403, 0.81, 1.3] : students
[1.0093, 0.2839, 0.22, 1.8998] : are
[1.0411, -0.69, 0.63, 1.1996] : brilliant


### Step 7: Query, Key and Value Matrices

In [23]:
WQ = [
    [1, 0],
    [0, 1],
    [1, 0],
    [0, 1]
]
WK = [
    [1, 0],
    [0, 1],
    [0.5, 0],
    [0, 0.5]
]
WV = [
    [0.5, 0],
    [0, 1],
    [1, 0],
    [0, 0.5]
]
Q = matrix_multiply(Z, WQ)
K = matrix_multiply(Z, WK)
V = matrix_multiply(Z, WV)
print("Q Matrix:")
print_matrix(Q)
print("\nK Matrix:")
print_matrix(K)
print("\nV Matrix:")
print_matrix(V)

Q Matrix:
[ 0.3000 2.9000 ]
[ 2.2515 1.9403 ]
[ 1.2293 2.1837 ]
[ 1.6711 0.5096 ]

K Matrix:
[ 0.2500 2.1500 ]
[ 1.8465 1.2903 ]
[ 1.1193 1.2338 ]
[ 1.3561 -0.0902 ]

V Matrix:
[ 0.2000 2.1500 ]
[ 1.5307 1.2903 ]
[ 0.7246 1.2338 ]
[ 1.1506 -0.0902 ]


## Step 8: Query-Key Comaprison
S = QK^T

In [24]:
def transpose(matrix):
    rows = len(matrix)
    cols = len(matrix[0])
    result = []
    for j in range(cols):
        row = []
        for i in range(rows):
            row.append(matrix[i][j])
        result.append(row)
    return result
KT = transpose(K)
S = matrix_multiply(Q, KT)
print("Raw Attention Score Matrix S = QK^T:")
print_matrix(S)

Raw Attention Score Matrix S = QK^T:
[ 6.3100 4.2957 3.9137 0.1452 ]
[ 4.7344 6.6607 4.9139 2.8782 ]
[ 5.0022 5.0874 4.0700 1.4701 ]
[ 1.5133 3.7431 2.4991 2.2203 ]


### Step 9: Scale the Scores
S' = S / sqrt(d_k)

In [25]:
d_k = 2
scale = square_root(d_k)
S_scaled = []
for row in S:
    new_row = []
    for value in row:
        new_row.append(value / scale)
    S_scaled.append(new_row)
print("Scaled Attention Scores:")
print_matrix(S_scaled)

Scaled Attention Scores:
[ 4.4618 3.0376 2.7674 0.1027 ]
[ 3.3477 4.7099 3.4746 2.0352 ]
[ 3.5371 3.5973 2.8780 1.0395 ]
[ 1.0701 2.6468 1.7672 1.5700 ]


### Step 10: Softmax

In [26]:
def softmax(row):
    # Find maximum for numerical stability
    maximum = row[0]
    for value in row:
        if value > maximum:
            maximum = value
    # Calculate exponentials
    exp_values = []
    for value in row:
        exp_values.append(exponential(value - maximum))
    # Calculate sum
    total = sum(exp_values)
    # Normalize
    probabilities = []
    for value in exp_values:
        probabilities.append(value / total)
    return probabilities
A = []
for row in S_scaled:
    A.append(softmax(row))
print("Attention Weight Matrix A:")
print_matrix(A, 4)

Attention Weight Matrix A:
[ 0.6958 0.1675 0.1278 0.0089 ]
[ 0.1585 0.6189 0.1800 0.0427 ]
[ 0.3757 0.3990 0.1944 0.0309 ]
[ 0.1053 0.5096 0.2115 0.1736 ]


### Step 11: Attention Output

In [27]:
O = matrix_multiply(A, V)
print("Attention Output O = AV:")
print_matrix(O)

Attention Output O = AV:
[ 0.4984 1.8690 ]
[ 1.1585 1.3575 ]
[ 0.8624 1.5596 ]
[ 1.1541 1.1292 ]


### Step 12: Output Projection

In [28]:
WO = [
    [1, 0, 0, 1],
    [0, 1, 1, 0]
]
H = matrix_multiply(O, WO)
print("Output Projection H = OWO:")
print_matrix(H)

Output Projection H = OWO:
[ 0.4984 1.8690 1.8690 0.4984 ]
[ 1.1585 1.3575 1.3575 1.1585 ]
[ 0.8624 1.5596 1.5596 0.8624 ]
[ 1.1541 1.1292 1.1292 1.1541 ]


### Step 13: Residual Connection
R = Z + H

In [29]:
R = []
for i in range(len(Z)):
    row = []
    for j in range(d_model):
        row.append(Z[i][j] + H[i][j])
    R.append(row)
print("Residual Matrix R = Z + H:")
print_matrix(R)

Residual Matrix R = Z + H:
[ 0.6984 3.2690 1.9690 1.9984 ]
[ 2.6000 1.9978 2.1675 2.4585 ]
[ 1.8717 1.8435 1.7796 2.7622 ]
[ 2.1953 0.4392 1.7592 2.3537 ]


### Step 14: Layer Normalization

In [32]:
def layer_normalize(row):
    mean = sum(row) / len(row)
    variance = 0.0
    for value in row:
        variance += (value - mean) ** 2
    variance = variance / len(row)
    std = square_root(variance + 1e-8)
    normalized = []
    for value in row:
        normalized.append((value - mean) / std)
    return normalized, mean, variance
Y = []
print("Layer Normalization:")
for i in range(len(R)):
    normalized, mean, variance = layer_normalize(R[i])
    Y.append(normalized)
    print(tokens[i])
    print("Mean     :", round(mean, 4))
    print("Variance :", round(variance, 4))
    print("Output   :", [round(x, 4) for x in normalized])
    print()

Layer Normalization:
ai
Mean     : 1.9837
Variance : 0.8261
Output   : [-1.4141, 1.4141, -0.0162, 0.0162]

students
Mean     : 2.3059
Variance : 0.056
Output   : [1.243, -1.3026, -0.5852, 0.6448]

are
Mean     : 2.0642
Variance : 0.1635
Output   : [-0.4763, -0.546, -0.7039, 1.7262]

brilliant
Mean     : 1.6868
Variance : 0.5662
Output   : [0.6756, -1.658, 0.0962, 0.8862]



### Step 15: Feed-Forward Network

In [33]:
W1 = [
    [1, 0, 1],
    [0, 1, 1],
    [1, 1, 0],
    [1, -1, 1]
]
W2 = [
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [1, 1, 0, 1]
]
print("W1:")
print_matrix(W1)
print("\nW2:")
print_matrix(W2)

W1:
[ 1.0000 0.0000 1.0000 ]
[ 0.0000 1.0000 1.0000 ]
[ 1.0000 1.0000 0.0000 ]
[ 1.0000 -1.0000 1.0000 ]

W2:
[ 1.0000 0.0000 1.0000 0.0000 ]
[ 0.0000 1.0000 0.0000 1.0000 ]
[ 1.0000 1.0000 0.0000 1.0000 ]


GeLu Calculation

In [34]:
def gelu(x):
    pi = 3.141592653589793
    def tanh(value):
        e1 = exponential(2 * value)
        return (e1 - 1) / (e1 + 1)
    coefficient = (2 / pi) ** 0.5
    inner = coefficient * (x + 0.044715 * (x ** 3))
    return 0.5 * x * (1 + tanh(inner))

FFN(Y) = GELU(YW1)W2

In [36]:
# First linear transformation
hidden = matrix_multiply(Y, W1)
print("Hidden Layer YW1:")
print_matrix(hidden)

# Apply GELU
G = []
for row in hidden:
    new_row = []
    for value in row:
        new_row.append(gelu(value))
    G.append(new_row)
print("\nAfter GELU:")
print_matrix(G)

# Second linear transformation
FFN = matrix_multiply(G, W2)
print("\nFFN Output:")
print_matrix(FFN)

Hidden Layer YW1:
[ -1.4141 1.3818 0.0162 ]
[ 1.3026 -2.5326 0.5852 ]
[ 0.5460 -2.9760 0.7039 ]
[ 1.6580 -2.4480 -0.0962 ]

After GELU:
[ -0.1115 1.2661 0.0082 ]
[ 1.1768 -0.0139 0.4218 ]
[ 0.3862 -0.0039 0.5344 ]
[ 1.5772 -0.0172 -0.0444 ]

FFN Output:
[ -0.1033 1.2743 -0.1115 1.2743 ]
[ 1.5986 0.4079 1.1768 0.4079 ]
[ 0.9206 0.5305 0.3862 0.5305 ]
[ 1.5328 -0.0616 1.5772 -0.0616 ]


### Step 16: Second Residual Connection
R2 = Y + FFN(Y)

In [37]:
R2 = []
for i in range(len(Y)):
    row = []
    for j in range(d_model):
        row.append(Y[i][j] + FFN[i][j])
    R2.append(row)
print("Second Residual Matrix R2:")
print_matrix(R2)

Second Residual Matrix R2:
[ -1.5174 2.6884 -0.1276 1.2905 ]
[ 2.8416 -0.8946 0.5916 1.0527 ]
[ 0.4444 -0.0155 -0.3177 2.2566 ]
[ 2.2084 -1.7195 1.6733 0.8246 ]


### Step 17: Second Layer Normalization
Zout = LayerNorm(R2)

In [38]:
Zout = []
for row in R2:
    normalized, mean, variance = layer_normalize(row)
    Zout.append(normalized)
print("Final Encoder Output:")
print()
for i in range(len(Zout)):
    print(tokens[i], ":", [round(x, 4) for x in Zout[i]])

Final Encoder Output:

ai : [-1.3388, 1.3414, -0.4532, 0.4505]
students : [1.4581, -1.3445, -0.2297, 0.1162]
are : [-0.1478, -0.6083, -0.9108, 1.6669]
brilliant : [0.97, -1.6366, 0.6149, 0.0517]


# FINAL RESULT

In [43]:
print("=" * 60)
print("FINAL CONTEXT-AWARE REPRESENTATIONS")
print("=" * 60)
for i in range(len(tokens)):
    print(
        [round(value, 4) for value in Zout[i]],
        ":",
        tokens[i]
    )
print()
print("=" * 60)
print("Transformer Encoder completed successfully.")
print("=" * 60)

FINAL CONTEXT-AWARE REPRESENTATIONS
[-1.3388, 1.3414, -0.4532, 0.4505] : ai
[1.4581, -1.3445, -0.2297, 0.1162] : students
[-0.1478, -0.6083, -0.9108, 1.6669] : are
[0.97, -1.6366, 0.6149, 0.0517] : brilliant

Transformer Encoder completed successfully.


# TRANSFORMER ENCODER PIPELINE

In [44]:
print("""
RAW TEXT
   ↓
PREPROCESSING
   ↓
TOKENIZATION
   ↓
TOKEN IDs
   ↓
EMBEDDING LOOKUP
   ↓
POSITIONAL ENCODING
   ↓
Q, K, V
   ↓
QK^T
   ↓
SCALING
   ↓
SOFTMAX
   ↓
ATTENTION OUTPUT (AV)
   ↓
OUTPUT PROJECTION
   ↓
RESIDUAL CONNECTION
   ↓
LAYER NORMALIZATION
   ↓
FEED-FORWARD NETWORK
   ↓
GELU
   ↓
SECOND RESIDUAL CONNECTION
   ↓
SECOND LAYER NORMALIZATION
   ↓
FINAL CONTEXT-AWARE REPRESENTATION
""")


RAW TEXT
   ↓
PREPROCESSING
   ↓
TOKENIZATION
   ↓
TOKEN IDs
   ↓
EMBEDDING LOOKUP
   ↓
POSITIONAL ENCODING
   ↓
Q, K, V
   ↓
QK^T
   ↓
SCALING
   ↓
SOFTMAX
   ↓
ATTENTION OUTPUT (AV)
   ↓
OUTPUT PROJECTION
   ↓
RESIDUAL CONNECTION
   ↓
LAYER NORMALIZATION
   ↓
FEED-FORWARD NETWORK
   ↓
GELU
   ↓
SECOND RESIDUAL CONNECTION
   ↓
SECOND LAYER NORMALIZATION
   ↓
FINAL CONTEXT-AWARE REPRESENTATION

